In [ ]:
#!/usr/bin/env python3
# train_FG_long_k_fixed.py
# Fixed: removed undefined k_map usage and removed printing of model parameter count.
# Convert wide (ETA, F1..F12, G1..G12) -> long dataset with mapping k->(Ri,M)
# Train NN to predict (Fval,Gval) from inputs (Ri, M, Eta, k).
# Optional: you can later extend pde_residuals(...) and set USE_PDE=True for PINN.

import os, time, json
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split

# ---------------- User settings ----------------
DATA_PATH = "Grad-4-Merged.csv"   # your CSV
OUT_DIR = "Grad-4a-pinn-results"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cf -> (Epsilon,N) as you described
Cf_map = {
    1: (0, 10), 2: (0.1, 10), 3: (0.3, 10),
    4: (0.5, 10),  5: (0.7, 10)
}

# training hyperparams (tune)
SEED = 42
N_EPOCHS = 1600
BATCH_SIZE = 128
LR = 2e-3
ONECYCLE_MAX_LR = 8e-3
GRAD_CLIP = 5.0
LBFGS_ITERS = 200
EARLY_STOPPING_PATIENCE = 300
PRINT_EVERY = 80

USE_PDE = False   # set True only after you implement pde_residuals()
# ------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# ---------------- Load CSV and build long dataset ----------------
df_w = pd.read_csv(DATA_PATH)
print("Loaded CSV:", DATA_PATH, "shape:", df_w.shape)
print("Columns:", df_w.columns.tolist()[:20])

# find ETA column
eta_col = next((c for c in df_w.columns if c.strip().lower()=='eta' or 'eta' in c.lower()), None)
if eta_col is None:
    raise RuntimeError("ETA column not found in CSV.")

# check F/G columns exist
Cf_cols = [f"Cf{i}" for i in range(1,6)]

for c in Cf_cols: #+G_cols:
    if c not in df_w.columns:
        raise RuntimeError(f"Column {c} not found in CSV.")

# Build long records: for each k create rows with assigned Ri,M
records = []
n_eta = df_w.shape[0]
for k in range(1,6):
    epsilon, n  = Cf_map[k]
    Cfcol = f"Cf{k}"
    for i in range(n_eta):
        records.append({
            "Epsilon": epsilon,
            "N": n,
            "Eta": float(df_w.iloc[i][eta_col]),
            "k": float(k),  # kept only for grouping, NOT used as model input,           # include k as numeric (we'll normalize)
            "Cfval": float(df_w.iloc[i][Cfcol])
        })
df = pd.DataFrame.from_records(records)
print("Built long dataframe shape:", df.shape)
df.to_csv(os.path.join(OUT_DIR,"FG_long_raw.csv"), index=False)

In [ ]:
# ---------------- Prepare model inputs/outputs ----------------
# Inputs: [Epsilon, N, Eta]  (k removed from model inputs) -> Outputs: [Cfval]
X = df[["Epsilon","N","Eta"]].values.astype(np.float32)
Y = df[["Cfval"]].values.astype(np.float32)

# Normalize inputs per-column (important)
X_mean = X.mean(axis=0); X_std = X.std(axis=0) + 1e-12
Y_mean = Y.mean(axis=0); Y_std = Y.std(axis=0) + 1e-12
Xn = (X - X_mean)/X_std
Yn = (Y - Y_mean)/Y_std

# Train/val/test split (stratify by (Ri,M) combos so each combo appears)
df["combo"] = df["Epsilon"].astype(str) + "_" + df["N"].astype(str)
idx = np.arange(Xn.shape[0])
train_idx, test_idx = train_test_split(idx, test_size=0.18, random_state=SEED, stratify=df['combo'])
# from test split further into val/test
val_idx, test_idx = train_test_split(test_idx, test_size=0.5, random_state=SEED, stratify=df['combo'].iloc[test_idx])

X_train = Xn[train_idx]; Y_train = Yn[train_idx]
X_val   = Xn[val_idx];   Y_val   = Yn[val_idx]
X_test  = Xn[test_idx];  Y_test  = Yn[test_idx]

print("Train/Val/Test sizes:", X_train.shape[0], X_val.shape[0], X_test.shape[0])

# Convert to torch
device = DEVICE
X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32, device=device)
X_val_t   = torch.tensor(X_val, dtype=torch.float32, device=device)
Y_val_t   = torch.tensor(Y_val, dtype=torch.float32, device=device)
X_all_t   = torch.tensor(Xn, dtype=torch.float32, device=device)  # for LBFGS final refine / full predictions

In [ ]:
# ---------------- Model definition ----------------
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.l1 = nn.Linear(dim, dim)
        self.act = nn.SiLU()
        self.l2 = nn.Linear(dim, dim)
        nn.init.xavier_uniform_(self.l1.weight); nn.init.zeros_(self.l1.bias)
        nn.init.xavier_uniform_(self.l2.weight); nn.init.zeros_(self.l2.bias)
    def forward(self,x):
        r = self.act(self.l1(x))
        r = self.l2(r)
        return self.act(x + r)

class FGLongNet(nn.Module):
    def __init__(self, in_dim=3, hidden=256, n_blocks=12, out_dim=1):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.SiLU()]
        for _ in range(n_blocks):
            layers.append(ResidualBlock(hidden))
        self.trunk = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, out_dim)
        nn.init.xavier_uniform_(self.head.weight); nn.init.zeros_(self.head.bias)
    def forward(self,x):
        return self.head(self.trunk(x))

model = FGLongNet(in_dim=3, hidden=256, n_blocks=12, out_dim=1).to(device)
# (intentionally not printing parameter count)

# ---------------- Optimizer + Scheduler ----------------
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-6)
steps_per_epoch = max(1, int(np.ceil(X_train.shape[0] / float(BATCH_SIZE))))
total_steps = N_EPOCHS * steps_per_epoch
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=ONECYCLE_MAX_LR,
                                          total_steps=total_steps, pct_start=0.1, div_factor=10.0)

mse = nn.MSELoss()

In [ ]:
# ---------------- Training loop ----------------
best_val = 1e12; patience = 0
history = {'train':[], 'val':[]}
t0 = time.time()
for ep in range(1, N_EPOCHS+1):
    model.train()
    perm = np.random.permutation(X_train.shape[0])
    losses=[]
    for i in range(0, X_train.shape[0], BATCH_SIZE):
        idxb = perm[i:i+BATCH_SIZE]
        xb = torch.tensor(X_train[idxb], dtype=torch.float32, device=device)
        yb = torch.tensor(Y_train[idxb], dtype=torch.float32, device=device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = mse(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        try:
            scheduler.step()
        except Exception:
            pass
        losses.append(loss.item())
    train_loss = float(np.mean(losses))
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t); val_loss = float(mse(val_pred, Y_val_t).item())
    history['train'].append(train_loss); history['val'].append(val_loss)
    if ep % PRINT_EVERY == 0 or ep==1:
        print(f"Epoch {ep}/{N_EPOCHS} train={train_loss:.3e} val={val_loss:.3e} lr={optimizer.param_groups[0]['lr']:.2e}")
    if val_loss < best_val:
        best_val = val_loss; patience = 0
        torch.save(model.state_dict(), os.path.join(OUT_DIR,"best_long.pt"))
    else:
        patience += 1
        if patience > EARLY_STOPPING_PATIENCE:
            print("Early stopping at epoch", ep); break

t1 = time.time()
print("Training done in {:.1f}s best_val={:.4e}".format(t1-t0, best_val))
model.load_state_dict(torch.load(os.path.join(OUT_DIR,"best_long.pt"), map_location=device))
model.eval()

In [ ]:
# LBFGS refine (optional)
print("LBFGS refine...")
try:
    lbfgs = optim.LBFGS(model.parameters(), max_iter=LBFGS_ITERS)
    def closure():
        lbfgs.zero_grad()
        pred = model(X_all_t)
        loss = mse(pred, torch.tensor(Yn, dtype=torch.float32, device=device))
        loss.backward()
        return loss
    lbfgs.step(closure)
except Exception as e:
    print("LBFGS skipped/failed:", e)

# ---------------- Predict back to original scale ----------------
with torch.no_grad():
    Ypred_n = model(X_all_t).cpu().numpy()
Ypred = Ypred_n * Y_std + Y_mean   # invert normalization

# attach predictions to df and save
df['p_Cfval'] = Ypred[:,0]

out_csv = os.path.join(OUT_DIR, "FG_long_preds.csv")
df.to_csv(out_csv, index=False)
print("Saved long-format predictions to:", out_csv)

# Compute RMSE overall and per combo and per k
df['err_Cf2'] = (df['p_Cfval'] - df['Cfval'])**2

rmse_overall = np.sqrt(df[['err_Cf2']].mean(axis=0))
print("Overall RMSE (Cf):", rmse_overall.tolist())

# per k RMSE
rmse_k = df.groupby('k').apply(lambda g: np.sqrt(np.mean((g['p_Cfval']-g['Cfval'])**2)))

rmse_df = pd.DataFrame({'k': rmse_k.index, 'rmse_Cf': rmse_k.values})
rmse_df.to_csv(os.path.join(OUT_DIR,"rmse_per_k.csv"), index=False)
print("Saved rmse_per_k.csv")

In [ ]:
# ---------------- plots: reconstruct curves per k (Ri,M) ----------------
for k in sorted(df['k'].unique()):
    sub = df[df['k']==k].sort_values('Eta')
    # use Cf_map (or G_map) to recover Ri,M for plotting
    epsilon, n = Cf_map[int(k)]
    plt.figure(figsize=(8,6))
    plt.plot(sub['Eta'], sub['Cfval'], '-', label='F true')
    plt.plot(sub['Eta'], sub['p_Cfval'], '--', label='F pred')
    
    plt.title(f"k={int(k)} Epsilon={epsilon} N={n}")
    plt.legend()
    plt.xlabel("Eta")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"curve_k{int(k)}_Epsilon{epsilon}_N{n}.png"))
    plt.close()

print("Saved curve plots and parity/rmse info in", OUT_DIR)


In [ ]:
#Plot the training history
plt.figure(figsize=(8,6))
plt.plot(history['train'], label='Train Loss')
plt.plot(history['val'], label='Val Loss')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
#PLOT PLOTLY GRAPH FOR aCTUAL VALUES F1 to F12
import plotly.express as px
import plotly.graph_objects as go
fig = go.Figure()
for k in range(1,6):
    sub = df[df['k']==k].sort_values('Eta')
    epsilon, n = Cf_map[int(k)]
    fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['Cfval'], mode='lines', name=f'F true Epsilon={epsilon} N={n}'))
    #fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['p_Fval'], mode='lines+markers', name=f'F pred k={int(k)} Epsilon={epsilon} N={n}'))
    #fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['Gval'], mode='lines+markers', name=f'G true k={int(k)} Epsilon={epsilon} N={n}'))
    #fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['p_Gval'], mode='lines+markers', name=f'G pred k={int(k)} Epsilon={epsilon} N={n}'))
fig.update_layout(title='F and G values vs Eta for different k, Epsilon, N', xaxis_title='Eta', yaxis_title='Values')
fig.write_html(os.path.join(OUT_DIR, "F_values_plotly.html"))
fig.show()

In [ ]:
#PLOT PLOTLY GRAPH FOR aCTUAL VALUES F1 to F12
import plotly.express as px
import plotly.graph_objects as go
fig = go.Figure()
for k in range(1,6):
    sub = df[df['k']==k].sort_values('Eta')
    epsilon, n = Cf_map[int(k)]
    #fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['Fval'], mode='lines+markers', name=f'F true k={int(k)} Ec={ec} Xi={xi}'))
    fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['p_Cfval'], mode='lines', name=f'F pred k={int(k)} Epsilon={epsilon} N={n}'))
    #fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['Gval'], mode='lines+markers', name=f'G true k={int(k)} Epsilon={epsilon} N={n}'))
    #fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['p_Gval'], mode='lines+markers', name=f'G pred k={int(k)} Epsilon={epsilon} N={n}'))
fig.update_layout(title='Cf values vs Eta for different Epsilon, N', xaxis_title='Eta', yaxis_title='Values')
fig.write_html(os.path.join(OUT_DIR, "Cf_Pred_values_plotly.html"))
fig.show()

In [ ]:
#PLOT PLOTLY GRAPH FOR aCTUAL VALUES F1 to F12
import plotly.express as px
import plotly.graph_objects as go
fig = go.Figure()
for k in range(1,6):
    sub = df[df['k']==k].sort_values('Eta')
    epsilon, n = Cf_map[int(k)]
    fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['Cfval'], mode='lines', name=f'Cf true k={int(k)} Epsilon={epsilon} N={n}'))
    fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['p_Cfval'], mode='markers', name=f'Cf pred k={int(k)} Epsilon={epsilon} N={n}'))
   
fig.update_layout(title='Cf values vs Eta for different k, Epsilon, N', xaxis_title='Eta', yaxis_title='Values')
fig.write_html(os.path.join(OUT_DIR, "Cf_Pred_values_plotly.html"))
fig.show()